# 01 - RDFLib Basics with the Human Phenotype Ontology (HPO)

## Learning objectives
- Load an OWL ontology file (`data/hp.owl`) with RDFLib.
- Inspect graph size and namespaces.
- Build a function that returns triples related to a class ID in Turtle format.
- Validate the function with class `HP_0100651`.

In [1]:
from pathlib import Path
from rdflib import Graph, URIRef

# Resolve project root robustly so the notebook works from different launch folders.
cwd = Path.cwd()
if (cwd / "data" / "hp.owl").exists():
    project_root = cwd
elif (cwd.parent / "data" / "hp.owl").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Could not find data/hp.owl from current working directory.")

hp_owl_path = project_root / "data" / "hp.owl"

# Parse the ontology into an RDFLib graph.
g = Graph()
g.parse(hp_owl_path)

print(f"Loaded ontology from: {hp_owl_path}")
print(f"Total triples: {len(g):,}")

Loaded ontology from: /Users/gianmariasilvello/Library/CloudStorage/Dropbox/Documents/Didattica/LLM-KB/pythonProject/LLM-KB/data/hp.owl
Total triples: 905,482


In [3]:
def resolve_class_uri(graph: Graph, class_id: str) -> URIRef:
    """
    Resolve a class ID (e.g., HP_0100651) to the full URI used in the graph.

    Strategy:
    1. Find any URI subject ending with the provided ID.
    2. If not found, also try replacing ':' with '_' (for IDs like HP:0100651).
    """
    normalized = class_id.replace(":", "_")

    # Search subjects first because class resources typically appear as subjects.
    for subject in set(graph.subjects()):
        if isinstance(subject, URIRef) and str(subject).endswith(class_id):
            return subject

    for subject in set(graph.subjects()):
        if isinstance(subject, URIRef) and str(subject).endswith(normalized):
            return subject

    raise ValueError(f"Class ID '{class_id}' not found in graph.")

In [4]:
def class_related_triples_as_turtle(graph: Graph, class_id: str, include_incoming: bool = True) -> str:
    """
    Return triples related to a class ID in Turtle format.

    Related triples include:
    - Outgoing triples: (class_uri, predicate, object)
    - Incoming triples: (subject, predicate, class_uri), optional
    """
    class_uri = resolve_class_uri(graph, class_id)
    subgraph = Graph()

    # Preserve namespace bindings for readable Turtle output.
    for prefix, namespace in graph.namespaces():
        subgraph.bind(prefix, namespace, override=False)

    # Add outgoing triples where the class is the subject.
    for predicate, obj in graph.predicate_objects(class_uri):
        subgraph.add((class_uri, predicate, obj))

    # Optionally add incoming triples where the class is the object.
    if include_incoming:
        for subject, predicate in graph.subject_predicates(class_uri):
            subgraph.add((subject, predicate, class_uri))

    if len(subgraph) == 0:
        raise ValueError(f"No related triples found for class ID '{class_id}'.")

    return subgraph.serialize(format="turtle")

In [5]:
# Testbed class requested in the assignment.
test_class_id = "HP_0100651"
ttl_output = class_related_triples_as_turtle(g, test_class_id)

print(f"Related triples for {test_class_id} (Turtle):\n")
print(ttl_output[:3000])  # Print a preview to keep output readable in class.
print("\n... (truncated preview)")

Related triples for HP_0100651 (Turtle):

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix oboInOwl: <http://www.geneontology.org/formats/oboInOwl#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

obo:HP_0100651 a owl:Class ;
    rdfs:label "Type I diabetes mellitus" ;
    obo:IAO_0000115 "A chronic condition in which the pancreas produces little or no insulin. Type I diabetes mellitus is manifested by the sudden onset of severe hyperglycemia with rapid progression to diabetic ketoacidosis unless treated with insulin." ;
    dcterms:creator <https://orcid.org/0009-0006-4530-3154> ;
    oboInOwl:creation_date "2010-12-29T06:37:55Z" ;
    oboInOwl:hasDbXref "SNOMEDCT_US:46635009",
        "UMLS:C0011854" ;
    oboInOwl:hasExactSynonym "Diabetes mellitus Type I",
        "Juvenile diabetes mellitus",
        "Type 1 diabetes",
        "Type I diabetes" ;
    oboInOwl:hasRelate

## Exercises
1. Modify the function to return only outgoing triples and compare the result size.
2. Add an optional filter to keep only selected predicates (e.g., `rdfs:label`, `oboInOwl:id`).
3. Adapt the method to return a Python list of triples instead of Turtle serialization.